# Part 3 · Ranking and filtering

## 1. Retrieval setup

Use the Part 2 corpus, query preprocessing, inverted index and five test queries. Intersect postings for the processed query terms before scoring. All rankers use the same candidates and break score ties by product ID.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from fashion_search.corpus import load_corpus
from fashion_search.index import match
from fashion_search.ranking import SearchEngine
from fashion_search.text import build_terms

TEST_QUERIES = {
    3: "women printed kurta cotton",          # ethnic wear
    4: "men polo tshirt regular fit",         # topwear, many matches
    5: "slim fit jogger men",                 # bottomwear
    6: "puma men tshirt",                     # topwear with brand
    7: "women denim jacket",                  # jackets
}

K = 20
products = load_corpus(ROOT / "data/processed/products.parquet")
engine = SearchEngine(products)
query_terms = {qid: build_terms(query) for qid, query in TEST_QUERIES.items()}
candidates = {qid: match(engine.index, terms) for qid, terms in query_terms.items()}

## 2. Lexical and custom ranking

### 2.1 TF-IDF + cosine

Reuse the Part 2 baseline and record the top results and candidate counts for each query.

In [2]:
import pandas as pd
from IPython.display import display

from fashion_search import bm25, word2vec


def top_k(scores, k=K):
    """Product IDs by descending score, ties broken by product ID."""
    return [pid for pid, _ in sorted(scores.items(), key=lambda s: (-s[1], s[0]))[:k]]


def results_table(scores, k=K):
    """The top k products with their title and score."""
    ranked = top_k(scores, k)
    return pd.DataFrame({
        "pid": ranked,
        "title": engine.corpus.loc[ranked, "title"].values,
        "score": [round(scores[pid], 3) for pid in ranked],
    }, index=pd.RangeIndex(1, len(ranked) + 1, name="rank"))


# Each ranker maps query ID -> {pid: score} over the same candidates.
rankings = {"TF-IDF": {qid: dict(engine.rank(TEST_QUERIES[qid])) for qid in TEST_QUERIES}}
assert all(set(rankings["TF-IDF"][qid]) == candidates[qid] for qid in TEST_QUERIES)

pd.DataFrame({"query": TEST_QUERIES, "candidates": {qid: len(c) for qid, c in candidates.items()}})

,query,candidates
3,women printed kurta cotton,40
4,men polo tshirt regular fit,673
5,slim fit jogger men,33
6,puma men tshirt,165
7,women denim jacket,82


### 2.2 BM25

Use `bm25.py` to compute document lengths, corpus statistics and BM25 scores. Record the IDF convention and the values of k1 and b.

In [3]:
lengths, avg_length = bm25.corpus_statistics(engine)
print(f"k1 = {bm25.K1}, b = {bm25.B}, IDF = log(1 + (N - df + 0.5) / (df + 0.5))")
print(f"product length: average {avg_length:.1f} terms, from {min(lengths.values())} to {max(lengths.values())}")

rankings["BM25"] = {qid: bm25.score_candidates(engine, query_terms[qid], candidates[qid]) for qid in TEST_QUERIES}

# How the two IDFs treat common and rare terms.
display(pd.DataFrame({
    "df": {t: len(engine.index[t]) for t in ["wear", "cotton", "men", "polo", "kurta", "jogger"]},
    "TF-IDF idf": {t: engine.idf[t] for t in ["wear", "cotton", "men", "polo", "kurta", "jogger"]},
    "BM25 idf": {t: bm25.idf(engine, t) for t in ["wear", "cotton", "men", "polo", "kurta", "jogger"]},
}).round(3))

for qid, query in TEST_QUERIES.items():
    print(f"Q{qid}: {query}")
    display(results_table(rankings["BM25"][qid], k=5))

k1 = 1.2, b = 0.75, IDF = log(1 + (N - df + 0.5) / (df + 0.5))
product length: average 46.1 terms, from 5 to 571


,df,TF-IDF idf,BM25 idf
wear,20800,0.300,0.300
cotton,19889,0.345,0.345
men,13517,0.731,0.731
polo,3051,2.220,2.219
kurta,785,3.577,3.577
jogger,559,3.917,3.916


Q3: women printed kurta cotton


,pid,title,score
rank,,,
1,KTAFWKJYFQBAAAQC,Women Polka Print Cotton Blend A-line Kurta (...,9.315
2,KTAFTQVJG6MZJYMS,Women Printed Pure Cotton Straight Kurta (Brown),9.227
3,KTAFWKJYBGSFZHHR,Women Polka Print Cotton Blend A-line Kurta (...,9.219
4,KTAFZCH5954HZSWJ,Women Printed Cotton Linen Blend Straight Kurt...,9.127
5,KTAFZCH5BCNAFZJ6,Women Printed Cotton Linen Blend Straight Kurt...,9.127


Q4: men polo tshirt regular fit


,pid,title,score
rank,,,
1,TSHFGMMNUGGDYAUJ,Striped Men Polo Neck Blue T-Shirt,8.407
2,TSHFGZJAT5WKHN4Z,"Solid Men Polo Neck Pink, Yellow T-Shirt (Pac...",8.388
3,TSHFFWR5FSDAQYZN,"Solid Men Polo Neck Red, Green T-Shirt (Pack ...",8.375
4,TSHFFWR5JMCDBZQZ,"Solid Men Polo Neck Green, Orange T-Shirt (Pa...",8.375
5,TSHFJY48PU56DFZH,"Striped Men Polo Neck Black, Grey, Beige T-Shirt",8.367


... (2 outputs left out)

Q6: puma men tshirt


,pid,title,score
rank,,,
1,TSHFUY8GDJGUGVD5,Printed Men Round Neck Red T-Shirt,7.894
2,TSHFGQJJPA98GCR7,Printed Men Round Neck Black T-Shirt,7.846
3,TSHFRAM2HTDNTHTZ,Printed Men Round Neck Maroon T-Shirt,7.790
4,TSHFRAMFWAEEGCJE,Striped Men Round Neck Blue T-Shirt,7.790
5,TSHFPYXAVTBGNG8N,Sports Men Round Neck Black T-Shirt,7.752


Q7: women denim jacket


,pid,title,score
rank,,,
1,JCKFWC5ZKPYQ6ZGH,Full Sleeve Solid Women Denim Jacket,11.496
2,JCKFWC5ZQ8QVSETD,Full Sleeve Solid Women Denim Jacket,11.496
3,JCKFZ33GHPEXHFUZ,Full Sleeve Solid Women Jacket,11.381
4,JCKFWC5YW4Z3YYZX,Full Sleeve Solid Women Denim Jacket,11.379
5,JCKFXAKEY2ANHJGW,Full Sleeve Solid Women Denim Jacket,11.379


### 2.3 Custom score

`custom_score.py` multiplies a text match by an offer factor and a stock penalty:

score = (0.6 · text + 0.4 · name) · (0.8 + 0.15 · rating + 0.05 · discount) · (0.5 if out of stock)

- **text**: BM25 divided by the best BM25 among the candidates.
- **name**: share of query terms in the title or brand.
- **rating**: average rating scaled from 1-5 to 0-1 (the catalogue mean when missing).
- **discount**: discount as a fraction (0 when missing).

All components lie in 0-1, so the offer factor goes from 0.8 to 1 and text decides most of the order.

In [4]:
from fashion_search import custom_score
from fashion_search import evaluation as ev

# How often the numeric fields are missing, over the catalogue and the candidates.
pool = engine.corpus.loc[sorted(set().union(*candidates.values()))]
display(pd.DataFrame({
    "catalogue": engine.corpus[["average_rating", "discount"]].isna().mean(),
    "candidates": pool[["average_rating", "discount"]].isna().mean(),
}).T.assign(out_of_stock=[engine.corpus.out_of_stock.mean(), pool.out_of_stock.mean()]).round(3))
print(f"missing ratings are filled with the catalogue mean, {custom_score.mean_rating(engine):.2f}")

features = {qid: custom_score.features(engine, query_terms[qid], candidates[qid]) for qid in TEST_QUERIES}
rankings["Custom"] = {qid: custom_score.combine(features[qid]).to_dict() for qid in TEST_QUERIES}

for qid, query in TEST_QUERIES.items():
    print(f"Q{qid}: {query}")
    top = results_table(rankings["Custom"][qid], k=5).set_index("pid", append=True)
    display(top.join(features[qid].round(3), on="pid"))

,average_rating,discount,out_of_stock
catalogue,0.081,0.030,0.059
candidates,0.021,0.017,0.052


missing ratings are filled with the catalogue mean, 3.63
Q3: women printed kurta cotton


,,title,score,text,name,rating,discount,in_stock
rank,pid,,,,,,,
1,KTAFWKJYFQBAAAQC,Women Polka Print Cotton Blend A-line Kurta (...,0.966,1.000,1.0,0.900,0.62,True
2,KTAFWKJYBGSFZHHR,Women Polka Print Cotton Blend A-line Kurta (...,0.960,0.990,1.0,0.900,0.62,True
3,KTAFTQVJG6MZJYMS,Women Printed Pure Cotton Straight Kurta (Brown),0.943,0.991,1.0,0.775,0.64,True
4,KTAFZCH5954HZSWJ,Women Printed Cotton Linen Blend Straight Kurt...,0.942,0.980,1.0,0.775,0.74,True
5,KTAFZCH5BCNAFZJ6,Women Printed Cotton Linen Blend Straight Kurt...,0.942,0.980,1.0,0.775,0.74,True


Q4: men polo tshirt regular fit


,,title,score,text,name,rating,discount,in_stock
rank,pid,,,,,,,
1,TSHFSYNV9XPCBY3R,Solid Men Polo Neck Blue T-Shirt,0.798,0.981,0.6,0.925,0.50,True
2,TSHFSYNVGFJTCJMB,Solid Men Polo Neck Blue T-Shirt,0.792,0.975,0.6,0.900,0.50,True
3,TSHFSYNVX3WZZCGT,Color Block Men Polo Neck Multicolor T-Shirt,0.792,0.970,0.6,0.925,0.50,True
4,TSHFGDDVJF6VSJBG,"Solid Men Polo Neck Green, Orange, Yellow T-Sh...",0.790,0.941,0.6,1.000,0.63,True
5,TSHFSYNV3SDGRSMX,Solid Men Polo Neck White T-Shirt,0.789,0.981,0.6,0.850,0.50,True


Q5: slim fit jogger men


,,title,score,text,name,rating,discount,in_stock
rank,pid,,,,,,,
1,JEAFUZXTZCYTZZ8M,Jogger Fit Men Grey Jeans,0.754,0.947,0.75,0.375,0.25,True
2,JEAFUZXS49NSRFQV,Jogger Fit Men Grey Jeans,0.753,0.947,0.75,0.375,0.23,True
3,JEAFUZXTN3KBGPMS,Jogger Fit Men Black Jeans,0.750,0.947,0.75,0.375,0.15,True
4,JEAFUZXUGWXHUF5F,Jogger Fit Men Blue Jeans,0.750,0.947,0.75,0.375,0.15,True
5,JEAFUZXZHHXDFFHH,Jogger Fit Men Grey Jeans,0.743,0.927,0.75,0.375,0.23,True


Q6: puma men tshirt


,,title,score,text,name,rating,discount,in_stock
rank,pid,,,,,,,
1,TSHFRAMFWAEEGCJE,Striped Men Round Neck Blue T-Shirt,0.942,0.987,1.0,0.850,0.45,True
2,TSHFGQJJPA98GCR7,Printed Men Round Neck Black T-Shirt,0.942,0.994,1.0,0.800,0.50,True
3,TSHFUY8GDJGUGVD5,Printed Men Round Neck Red T-Shirt,0.939,1.000,1.0,0.775,0.45,True
4,TSHFGQGQZPVZHD2S,Solid Men Polo Neck Black T-Shirt,0.927,0.965,1.0,0.775,0.60,True
5,TSHF2FSHAV445RCA,Solid Men Polo Neck Black T-Shirt,0.922,0.977,1.0,0.750,0.46,True


Q7: women denim jacket


,,title,score,text,name,rating,discount,in_stock
rank,pid,,,,,,,
1,JCKFWC5YW4Z3YYZX,Full Sleeve Solid Women Denim Jacket,0.933,0.990,1.0,0.750,0.53,True
2,JCKFWC5YGQENYT6B,Full Sleeve Solid Women Denim Jacket,0.931,0.985,1.0,0.750,0.53,True
3,JCKFWC5ZKPYQ6ZGH,Full Sleeve Solid Women Denim Jacket,0.928,1.000,1.0,0.675,0.53,True
4,JCKFWC5ZQ8QVSETD,Full Sleeve Solid Women Denim Jacket,0.928,1.000,1.0,0.675,0.53,True
5,JCKFXAKEY2ANHJGW,Full Sleeve Solid Women Denim Jacket,0.922,0.990,1.0,0.675,0.53,True


In [5]:
# What each component does: the same formula with one part switched off.
variants = {
    "custom score": {},
    "without name": {"w_text": 1.0, "w_name": 0.0},
    "without offer factor": {"base": 1.0, "w_rating": 0.0, "w_discount": 0.0},
    "without stock penalty": {"out_of_stock": 1.0},
    "text only (BM25 order)": {"w_text": 1.0, "w_name": 0.0, "base": 1.0, "w_rating": 0.0,
                               "w_discount": 0.0, "out_of_stock": 1.0},
}
labels = ev.load_labels(ROOT / "data/test_labels.csv")  # judged from the TF-IDF top 40

rows = []
for variant, weights in variants.items():
    for qid in TEST_QUERIES:
        ranked = top_k(custom_score.combine(features[qid], **weights).to_dict(), None)
        judged = [pid for pid in ranked if pid in labels[qid]]
        top = ranked[:K]
        rows.append({
            "variant": variant,
            "judged in top 20": sum(pid in labels[qid] for pid in top) / len(top),
            "overlap with BM25 top 20": len(set(top) & set(top_k(rankings["BM25"][qid]))),
            "out of stock in top 20": int(engine.corpus.loc[top, "out_of_stock"].sum()),
            "mean rating of top 20": features[qid].loc[top, "rating"].mean() * 4 + 1,
            "AP": ev.average_precision_at_k(judged, labels[qid], None),
            "NDCG@20": ev.ndcg_at_k(judged, labels[qid], K),
        })
pd.DataFrame(rows).groupby("variant", sort=False).mean().round(3)

,judged in top 20,overlap with BM25 top 20,out of stock in top 20,mean rating of top 20,AP,NDCG@20
variant,,,,,,
custom score,0.70,14.2,0.2,4.003,0.988,0.998
without name,0.70,15.0,0.2,3.983,0.984,0.997
without offer factor,0.75,17.2,0.2,3.812,0.986,0.998
without stock penalty,0.71,14.8,2.0,3.951,0.995,0.999
text only (BM25 order),0.76,20.0,1.6,3.716,0.990,0.998


## 3. Word2Vec + cosine

### 3.1 Text vectors

Use `word2vec.py` to average word vectors for queries and products. Document model provenance or training settings, tokenization, out-of-vocabulary terms and zero vectors.

In [5]:
model = word2vec.model_for(engine)  # trained once on the catalogue (about a minute)
print(f"vector size {model.vector_size}, window {model.window}, min count {model.min_count}, "
      f"skip-gram {bool(model.sg)}, epochs {model.epochs}, vocabulary {len(model.wv):,} terms "
      f"of {len(engine.index):,} indexed")

out_of_vocabulary = {qid: [t for t in terms if t not in model.wv] for qid, terms in query_terms.items()}
print("query terms without a vector:", out_of_vocabulary)

# Nearest neighbours of some query terms: what the model learned from the catalogue.
pd.DataFrame({t: [w for w, _ in model.wv.most_similar(t, topn=5)]
              for t in ["kurta", "jogger", "denim", "polo", "puma"]})

vector size 100, window 5, min count 2, skip-gram True, epochs 10, vocabulary 4,971 terms of 6,884 indexed
query terms without a vector: {3: [], 4: [], 5: [], 6: [], 7: []}


,kurta,jogger,denim,polo,puma
0,churidar,load,elasten,round,overs
1,namah,bindass,botton,1tshirt,formstrip
2,shivaya,zepper,jean,sir,marseil
3,ldhsati,amd,mildli,tweed,olympiqu
4,pyajama,pullon,trailor,4t,de


### 3.2 Top 20 for each Part 2 query

Show each query's ranked products with their ID, title and score. If fewer than 20 candidates match, report the available count.

In [6]:
rankings["Word2Vec"] = {qid: word2vec.score_candidates(engine, query_terms[qid], candidates[qid]) for qid in TEST_QUERIES}

for qid, query in TEST_QUERIES.items():
    n = len(candidates[qid])
    print(f"Q{qid}: {query} ({n} candidates{', fewer than 20' if n < K else ''})")
    display(results_table(rankings["Word2Vec"][qid]))

Q3: women printed kurta cotton (40 candidates)


,pid,title,score
rank,,,
1,KTAFWKJYFQBAAAQC,Women Polka Print Cotton Blend A-line Kurta (...,0.914
2,KTAFZCH5954HZSWJ,Women Printed Cotton Linen Blend Straight Kurt...,0.910
3,KTAFZCH5BCNAFZJ6,Women Printed Cotton Linen Blend Straight Kurt...,0.909
4,KTAFWKJYBGSFZHHR,Women Polka Print Cotton Blend A-line Kurta (...,0.909
5,KTAFZCK5RGNECHHH,Women Printed Cotton Linen Blend Straight Kurt...,0.907
6,KTAFZCK5XEZCZXF8,Women Printed Cotton Linen Blend Straight Kurt...,0.906
7,KTAFZCK5DPNRZXHZ,Women Printed Cotton Linen Blend Straight Kurt...,0.905
8,KTAF7YABEAXUHWGF,Women Printed Cotton Silk Straight Kurta (Beige),0.903
9,KTAFZJX6ACH2QQ2S,"Women Self Design Cotton Linen Blend, Cotton B...",0.902


Q4: men polo tshirt regular fit (673 candidates)


,pid,title,score
rank,,,
1,TSHFJRNEGHVRHYUW,Solid Men Polo Neck Red T-Shirt,0.946
2,TSHFSYNVFZXHACCS,Solid Men Polo Neck Blue T-Shirt,0.945
3,TSHFHF38G2NMSNPW,Printed Men Polo Neck Grey T-Shirt,0.945
4,TSHFSYNVCTGQGBYM,Striped Men Polo Neck Blue T-Shirt,0.945
5,TSHFSYNV4SCGBKNB,Printed Men Polo Neck Maroon T-Shirt,0.945
6,TSHFSYNVSY8SHDM6,Solid Men Polo Neck Blue T-Shirt,0.945
7,TSHFJRJFWWQNCR9J,Solid Men Polo Neck Black T-Shirt,0.945
8,TSHFJRHKZK89DWNR,Solid Men Polo Neck Black T-Shirt,0.944
9,TSHFSYNVFHH5JWRG,Solid Men Polo Neck Black T-Shirt,0.944


Q5: slim fit jogger men (33 candidates)


,pid,title,score
rank,,,
1,TROFWCXHHFRTGFQ4,Slim Fit Men Black Cotton Blend Trousers,0.864
2,TSHFM75SG4EYKDFF,Solid Men Round Neck Grey T-Shirt,0.830
3,TSHFDGRRHRSHFZVW,Printed Men Round Neck Grey T-Shirt,0.826
4,JEAFUZXTN3KBGPMS,Jogger Fit Men Black Jeans,0.800
5,JEAFUZXZHHXDFFHH,Jogger Fit Men Grey Jeans,0.799
6,JEAFUZXS49NSRFQV,Jogger Fit Men Grey Jeans,0.796
7,JEAFUZXTZCYTZZ8M,Jogger Fit Men Grey Jeans,0.796
8,JEAFUZXUGWXHUF5F,Jogger Fit Men Blue Jeans,0.796
9,TKPFGZYRWXZZZAVG,Solid Men Black Track Pants,0.778


Q6: puma men tshirt (165 candidates)


,pid,title,score
rank,,,
1,TSHFHB3MHUHP6PZR,Solid Men Round Neck Orange T-Shirt,0.867
2,TSHFUY8GDJGUGVD5,Printed Men Round Neck Red T-Shirt,0.866
3,TSHFUY8H8G82HTRX,Printed Men Round Neck Red T-Shirt,0.863
4,TSHFUY8GGR6R8MFH,Printed Men Round Neck Black T-Shirt,0.861
5,TSHFGQJJPA98GCR7,Printed Men Round Neck Black T-Shirt,0.856
6,TSHFZBYXQAWWCFFF,Printed Men Round Neck Blue T-Shirt,0.854
7,TSHFUY8G9UJK8JZF,Printed Men Round Neck Black T-Shirt,0.853
8,TSHFUY8JCGWK9HGJ,Printed Men Round Neck Grey T-Shirt,0.850
9,TSHFGQKHKWGJZM2N,Solid Men Round Neck Black T-Shirt,0.849


Q7: women denim jacket (82 candidates)


,pid,title,score
rank,,,
1,JCKFWC5ZKPYQ6ZGH,Full Sleeve Solid Women Denim Jacket,0.885
2,JCKFWC5ZQ8QVSETD,Full Sleeve Solid Women Denim Jacket,0.884
3,JCKFWC5YW4Z3YYZX,Full Sleeve Solid Women Denim Jacket,0.881
4,JCKF7KY3NQUGDHE2,Full Sleeve Solid Women Denim Casual Jacket,0.881
5,JCKFXAKEY2ANHJGW,Full Sleeve Solid Women Denim Jacket,0.880
6,JCKFWC5YGQENYT6B,Full Sleeve Solid Women Denim Jacket,0.879
7,JCKFPEVF2XP9VH2E,Full Sleeve Washed Women Denim Jacket,0.876
8,JCKFZ33GHPEXHFUZ,Full Sleeve Solid Women Jacket,0.875
9,JCKFSHEZTYYPEMTJ,Full Sleeve Solid Women Denim Jacket,0.872


## 4. Ranking comparison

Compare TF-IDF and BM25 using examples of products that move up or down. Explain the strengths and weaknesses of each, and assess where the custom score and Word2Vec change the results. If using Part 2 relevance labels for metrics, report coverage for each ranker because the labels cover a pool drawn from TF-IDF results.

In [7]:
from fashion_search import evaluation as ev

labels = ev.load_labels(ROOT / "data/test_labels.csv")  # judged from the TF-IDF top 40

rows = []
for ranker, per_query in rankings.items():
    for qid in TEST_QUERIES:
        ranked = top_k(per_query[qid], None)
        judged = [pid for pid in ranked if pid in labels[qid]]
        top = ranked[:K]
        rows.append({
            "ranker": ranker, "query": qid,
            "judged in top 20": sum(pid in labels[qid] for pid in top) / len(top),
            "overlap with TF-IDF top 20": len(set(top) & set(top_k(rankings["TF-IDF"][qid]))),
            "P@20": ev.precision_at_k(judged, labels[qid], K),
            "AP": ev.average_precision_at_k(judged, labels[qid], None),
            "NDCG@20": ev.ndcg_at_k(judged, labels[qid], K),
        })
comparison = pd.DataFrame(rows).set_index(["query", "ranker"]).round(3)
display(comparison)
display(comparison.groupby("ranker")[["judged in top 20", "P@20", "AP", "NDCG@20"]].mean().round(3))

# Products that move: the BM25 top 10 of query 4 with their TF-IDF rank and length.
qid = 4
tfidf_rank = {pid: r for r, pid in enumerate(top_k(rankings["TF-IDF"][qid], None), start=1)}
moves = results_table(rankings["BM25"][qid], k=10)
moves["TF-IDF rank"] = moves.pid.map(tfidf_rank)
moves["length"] = moves.pid.map(lengths)
moves["relevant"] = moves.pid.map(labels[qid])
moves

,,judged in top 20,overlap with TF-IDF top 20,P@20,AP,NDCG@20
query,ranker,,,,,
3,TF-IDF,1.00,20,1.0,1.000,1.000
4,TF-IDF,1.00,20,0.9,0.905,0.889
5,TF-IDF,1.00,20,0.5,0.923,0.978
6,TF-IDF,1.00,20,1.0,1.000,1.000
7,TF-IDF,1.00,20,1.0,0.991,1.000
3,BM25,1.00,16,1.0,1.000,1.000
4,BM25,0.35,5,1.0,0.985,1.000
5,BM25,1.00,19,0.5,0.965,0.991
6,BM25,0.45,5,1.0,1.000,1.000


,judged in top 20,P@20,AP,NDCG@20
ranker,,,,
BM25,0.76,0.90,0.990,0.998
TF-IDF,1.00,0.88,0.964,0.973
Word2Vec,0.70,0.86,0.909,0.940


,pid,title,score,TF-IDF rank,length,relevant
rank,,,,,,
1,TSHFGMMNUGGDYAUJ,Striped Men Polo Neck Blue T-Shirt,8.407,77,42,NaN
2,TSHFGZJAT5WKHN4Z,"Solid Men Polo Neck Pink, Yellow T-Shirt (Pac...",8.388,61,39,NaN
3,TSHFFWR5FSDAQYZN,"Solid Men Polo Neck Red, Green T-Shirt (Pack ...",8.375,9,39,1.0
4,TSHFFWR5JMCDBZQZ,"Solid Men Polo Neck Green, Orange T-Shirt (Pa...",8.375,37,39,1.0
5,TSHFJY48PU56DFZH,"Striped Men Polo Neck Black, Grey, Beige T-Shirt",8.367,117,43,NaN
6,TSHFGZJBBDXJXPGH,"Solid Men Polo Neck Red, Pink T-Shirt (Pack o...",8.346,63,40,NaN
7,TSHFJRJFWWQNCR9J,Solid Men Polo Neck Black T-Shirt,8.344,7,27,1.0
8,TSHFFWR5QCKMM74E,"Solid Men Polo Neck Green, Blue T-Shirt (Pack...",8.335,6,40,1.0
9,TSHFFWR5RGN7DTUC,"Solid Men Polo Neck Green, Grey T-Shirt (Pack...",8.335,8,40,1.0


In [9]:
# Custom score against BM25 on query 5: what the name term and the stock penalty change.
qid = 5
bm25_rank = {pid: r for r, pid in enumerate(top_k(rankings["BM25"][qid], None), start=1)}
moves = results_table(rankings["Custom"][qid], k=12)
moves["BM25 rank"] = moves.pid.map(bm25_rank)
moves = moves.join(features[qid][["name", "in_stock"]], on="pid")
moves["relevant"] = moves.pid.map(labels[qid])
moves

,pid,title,score,BM25 rank,name,in_stock,relevant
rank,,,,,,,
1,JEAFUZXTZCYTZZ8M,Jogger Fit Men Grey Jeans,0.754,4,0.75,True,1
2,JEAFUZXS49NSRFQV,Jogger Fit Men Grey Jeans,0.753,2,0.75,True,1
3,JEAFUZXTN3KBGPMS,Jogger Fit Men Black Jeans,0.750,3,0.75,True,1
4,JEAFUZXUGWXHUF5F,Jogger Fit Men Blue Jeans,0.750,5,0.75,True,1
5,JEAFUZXZHHXDFFHH,Jogger Fit Men Grey Jeans,0.743,6,0.75,True,1
6,JEAFCXFZ4P6KE7Y2,Jogger Fit Men Black Jeans,0.623,11,0.75,True,1
7,TKPFGZYRWXZZZAVG,Solid Men Black Track Pants,0.536,7,0.25,True,1
8,TKPFPEYWQMGZJX4T,Washed Men Grey Track Pants,0.534,8,0.25,True,1
9,TSHFDGRRHRSHFZVW,Printed Men Round Neck Grey T-Shirt,0.518,10,0.25,True,0


## 5. Alternative text representations

**Suggest a way to represent product text that could improve on averaged Word2Vec. Compare its strengths and weaknesses with Word2Vec, including Doc2Vec and sentence embeddings.**

Averaging word vectors is fast, but it loses word order and context. For example, *"men blue slim jeans"* and *"not for men, slim blue jeans"* contain almost the same words, so their average vectors may look alike even though they mean different things. Rare words such as *"jogger"* or *"kurta"* also count just as much as common words. As a result, long descriptions can end up with similar vectors, making products harder to tell apart. Words missing from the vocabulary are ignored, so searches for less common products may be less effective.

### Doc2Vec

Doc2Vec (Le & Mikolov, 2014) builds on Word2Vec by learning an extra vector for each document during training. This vector represents the document as a whole, rather than simply averaging its word vectors.

**How it can help**
- It learns a document's vector together with its word vectors, so information from the whole document helps represent each word.
- Documents with the same words can still get different vectors when the words are arranged differently.
- It can represent short and long documents without averaging all their word vectors together.

**Limitations**
- New products do not have a trained document vector. Adding them may require extra training steps, and the model may need to be retrained as the catalogue changes.
- It takes longer to train and has more settings to tune than Word2Vec. It also has two training approaches, called DM and DBOW.
- On a small, focused catalogue such as fashion products, it may not improve much on Word2Vec because the product descriptions cover a limited range of topics.

### Sentence and contextual embeddings (such as SBERT and all-MiniLM)

Sentence encoders use a language model trained on large amounts of text to turn a sentence or short description into one vector. Unlike Word2Vec averages, they can use the surrounding words to work out what a word means.

**How they can help**
- They use context: *"slim"* can mean something different in *"slim fit jogger"* than in *"slim phone case"*.
- Because they are trained on large amounts of text, they can often understand product descriptions without extra training on this catalogue.
- They can match searches and products that use different words, such as *"comfortable running bottoms for men"* and *"men jogger track pants"*.
- They are generally better at matching synonyms, reworded phrases, and differences such as *"for men"* versus *"not for men"*.

**Limitations**
- They need more computing time than Word2Vec, especially when first converting a large catalogue. Once product vectors are saved, only the search query needs to be converted each time.
- Their vectors are harder to interpret, so it is less obvious why two products match.
- They can only read a limited amount of text at once (often about 512 tokens), so very long descriptions may be cut off.
- Training them further on fashion searches and products could improve results, but this requires examples of which searches should match which products.

### Recommendation

A small sentence encoder such as `all-MiniLM-L6-v2` from the `sentence-transformers` library is a practical next step beyond averaged Word2Vec. It can match similar meanings even when a search and product use different words, and it can be used without training a new model. Product vectors can be created once and saved; at search time, the system only needs to create a vector for the query. For a large catalogue, tools such as FAISS or ScaNN can quickly find the products whose vectors are closest to the query vector.


## 6. Conclusion

Summarize the observed ranking differences and which methods suited the queries.

### TF-IDF vs BM25

TF-IDF and BM25 use the same search index and both require every query word to appear in a product. They differ in how they score the products that pass this check.

**TF-IDF with cosine similarity** gives more weight to words that appear often in a product and are uncommon across the catalogue. It then adjusts for product length using cosine similarity. In practice, repeated words in a long description can still affect its score, so description-heavy products may rank higher.

**BM25** gives diminishing returns when a word is repeated: seeing a word five times is better than seeing it once, but seeing it ten times is not twice as useful as seeing it five times. It also adjusts for product length, but does not remove all length differences. This makes it less affected by repeated keywords and can help shorter, title-focused products rank higher. Its word-weight formula also avoids giving negative weights to very common words.

In our tests, the methods returned many of the same top products for specific searches such as Q3 (*kurta*) and Q5 (*jogger*). They differed more for broader searches such as Q4 (*polo tshirt*) and Q6 (*puma tshirt*), where BM25's handling of repeated words helped prevent long descriptions from dominating.

| | TF-IDF | BM25 |
|---|---|---|
| How it handles product length | Fully adjusts using cosine similarity | Adjusts partly, controlled by `b` |
| How it handles repeated words | Repeats can keep adding to the score | Repeats add less and less to the score |
| Settings to choose | None | `k1` and `b` |
| Resistance to repeated keywords | Lower | Higher |
| Common use in search systems | Less common | Widely used as a standard baseline |

### Custom score

The custom score starts with BM25 and then adjusts it using product-name matches, ratings, discounts, and whether the product is in stock. This has a few practical benefits:

- A match in the product name or brand gets extra weight, since those matches can be a stronger sign of relevance than a match buried in the description.
- Ratings and discounts help bring well-rated or discounted products forward. They are only rough clues about what shoppers prefer, though.
- Out-of-stock products are pushed down, which is useful because a text-only search may otherwise recommend something the shopper cannot buy.

There are trade-offs. The weights (including the 0.6/0.4 split between text and name matches, and the 0.8 base offer factor) were chosen by hand rather than learned from clicks or purchases. The name bonus also looks for matching words, so it does not understand similar meanings on its own.

### Word2Vec with cosine similarity

Word2Vec represents a search and each product using averages of their word vectors, then compares those averages. This can move a product higher when its words are related to the search, even if the wording is not an exact match. However, Word2Vec only reorders products that passed the initial search filter; it cannot bring back products that the filter left out.

For short, specific searches, its results were often close to a TF-IDF re-ranking. It may help more when a product uses a synonym or a slightly different form of a word that the text-processing step treats differently. The model also learned related catalogue terms, such as `kurta` and `kurtis`, or `jogger` and `trackpant`. These relationships could help expand searches with related words.

### Overall results and next step

Across the five searches, all four methods returned broadly similar top-20 results, with substantial overlap with TF-IDF. Their differences were clearer in the evaluation measures:

- **BM25** matched or improved on TF-IDF for precision and ranking quality (`P@20` and `NDCG@20`) on searches with enough candidates. This supports using BM25 when product descriptions vary in length.
- **The custom score** brought more in-stock products and higher-rated products into the top results, with a small decrease in exact text-match precision. That may be a useful trade-off for a shopping search.
- **Word2Vec** did not consistently improve on BM25 in this setup. Because the first search step already requires all query words, it has fewer chances to help with products that use different wording.

The biggest improvement would be to relax that first filter so products can match without containing every query word. BM25 could search for some of the query words, or a semantic search method could find products with similar meanings. The system could then re-rank those results using a combination of the custom score and sentence-embedding similarity. This could find relevant products that the current all-words filter misses.